In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# Make the notebook runnable from either the repository root or this folder.
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "tinytorch").is_dir()
)
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

# Data loader experiments

Verify `TensorDataset` first, then exercise `DataLoader` as a complete batching pipeline. Every requirement is enforced with an assertion, so execution stops at the first regression. The final checklist is printed only after all checks pass.

In [3]:
import random

import numpy as np

from tinytorch.data_loader import DataLoader, TensorDataset
from tinytorch.tensor import Tensor_CP


def assert_raises(exception_type, operation):
    """Assert that operation raises exactly the expected exception family."""
    try:
        operation()
    except exception_type as error:
        return error
    except Exception as error:
        raise AssertionError(
            f"Expected {exception_type.__name__}, got {type(error).__name__}"
        ) from error
    raise AssertionError(f"Expected {exception_type.__name__}, but nothing was raised")


loader_checks = {}

## TensorDataset

A tensor dataset stores aligned tensors and returns one tuple of tensors per sample. These small checks establish the input contract used by the loader experiments below.

In [4]:
features = Tensor_CP([[1, 2], [3, 4], [5, 6], [7, 8]])
labels = Tensor_CP([0, 1, 1, 0])
dataset = TensorDataset(features, labels)

assert len(dataset) == 4  # samples, not number of stored tensors
first_sample = dataset[0]
assert isinstance(first_sample, tuple)
assert len(first_sample) == 2
assert all(isinstance(value, Tensor_CP) for value in first_sample)
assert np.array_equal(first_sample[0].data, np.array([1, 2], dtype=np.float32))
assert first_sample[1].data.item() == 0
print("PASS: four aligned samples return feature-label Tensor_CP pairs")

PASS: four aligned samples return feature-label Tensor_CP pairs


### Invalid TensorDataset inputs

Stored tensors must have matching first dimensions. Plain lists, NumPy arrays, and scalar tensors are rejected because they do not satisfy the dataset contract.

In [5]:
assert_raises(
    ValueError,
    lambda: TensorDataset(Tensor_CP([[1], [2], [3], [4]]), Tensor_CP([0, 1, 2])),
)
assert_raises(TypeError, lambda: TensorDataset([1, 2, 3, 4]))
assert_raises(TypeError, lambda: TensorDataset(np.array([1, 2, 3, 4])))
assert_raises(ValueError, lambda: TensorDataset(Tensor_CP(42)))
print("PASS: mismatched lengths, non-tensors, and scalar tensors are rejected")

PASS: mismatched lengths, non-tensors, and scalar tensors are rejected


### Empty and single-tensor datasets

In [6]:
empty_features = Tensor_CP(np.empty((0, 3), dtype=np.float32))
empty_labels = Tensor_CP(np.empty((0,), dtype=np.float32))
empty_dataset = TensorDataset(empty_features, empty_labels)
assert len(empty_dataset) == 0

single_dataset = TensorDataset(Tensor_CP([10, 20, 30, 40]))
assert len(single_dataset) == 4
single_sample = single_dataset[0]
assert isinstance(single_sample, tuple) and len(single_sample) == 1
assert isinstance(single_sample[0], Tensor_CP)
assert single_sample[0].data.item() == 10
print("PASS: aligned empty tensors and a single tensor both work")

PASS: aligned empty tensors and a single tensor both work


## DataLoader

The main fixture has 11 samples. Each feature begins with a unique sample ID, and its label follows `label = 3 * sample_id + 5`. That relationship lets us detect any feature-label misalignment after shuffling.

In [7]:
sample_ids = np.arange(11, dtype=np.float32)
feature_values = np.column_stack((sample_ids, sample_ids * 10 + 1))
label_values = sample_ids * 3 + 5
batch_dataset = TensorDataset(Tensor_CP(feature_values), Tensor_CP(label_values))

print(f"Samples:       {len(batch_dataset)}")
print(f"Feature shape: {feature_values.shape}")
print(f"Label shape:   {label_values.shape}")

Samples:       11
Feature shape: (11, 2)
Label shape:   (11,)


### Batch count, sizes, and shapes

With 11 samples and `batch_size=4`, ceiling division gives three batches. The final incomplete batch must be retained, so the first dimensions are `4`, `4`, and `3`. Collation adds that batch dimension without changing each sample's remaining dimensions.

In [8]:
sequential_loader = DataLoader(batch_dataset, batch_size=4, shuffle=False)
sequential_batches = list(sequential_loader)

batch_sizes = [batch_features.shape[0] for batch_features, _ in sequential_batches]
feature_shapes = [batch_features.shape for batch_features, _ in sequential_batches]
label_shapes = [batch_labels.shape for _, batch_labels in sequential_batches]

assert batch_sizes == [4, 4, 3]
assert len(sequential_loader) == 3
assert feature_shapes == [(4, 2), (4, 2), (3, 2)]
assert label_shapes == [(4,), (4,), (3,)]

loader_checks["11 samples with batch size 4 produces batch sizes 4, 4, 3"] = True
loader_checks["len(loader) returns 3"] = True
loader_checks["Batch feature and label shapes are correct"] = True

print(f"Batch sizes:    {batch_sizes}")
print(f"Feature shapes: {feature_shapes}")
print(f"Label shapes:   {label_shapes}")
print(f"len(loader):    {len(sequential_loader)}")
print("PASS: full and partial batches have the expected sizes and shapes")

Batch sizes:    [4, 4, 3]
Feature shapes: [(4, 2), (4, 2), (3, 2)]
Label shapes:   [(4,), (4,), (3,)]
len(loader):    3
PASS: full and partial batches have the expected sizes and shapes


### Sequential order with `shuffle=False`

When shuffling is disabled, concatenating the sample IDs from all batches must reproduce the dataset's original order exactly.

In [9]:
sequential_order = np.concatenate(
    [batch_features.data[:, 0] for batch_features, _ in sequential_batches]
).astype(int)

assert np.array_equal(sequential_order, np.arange(11))
loader_checks["shuffle=False preserves sequential order"] = True

print(f"Observed sample IDs: {sequential_order.tolist()}")
print("PASS: shuffle=False preserves sequential order")

Observed sample IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
PASS: shuffle=False preserves sequential order


### Shuffling keeps complete samples aligned

The loader shuffles dataset indices, not tensors independently. We run two epochs and verify both the feature-label relationship and exact once-per-epoch coverage. A fixed seed makes this notebook's displayed orders reproducible.

In [10]:
random.seed(7)
shuffled_loader = DataLoader(batch_dataset, batch_size=4, shuffle=True)
epoch_orders = []

for epoch in range(2):
    order = []
    for batch_features, batch_labels in shuffled_loader:
        ids = batch_features.data[:, 0]
        expected_labels = ids * 3 + 5
        assert np.array_equal(batch_labels.data, expected_labels)
        order.extend(ids.astype(int).tolist())

    unique_ids, counts = np.unique(order, return_counts=True)
    assert np.array_equal(unique_ids, np.arange(11))
    assert np.array_equal(counts, np.ones(11, dtype=int))
    epoch_orders.append(order)

assert epoch_orders[0] != list(range(11))
assert epoch_orders[0] != epoch_orders[1]
loader_checks["shuffle=True preserves feature-label alignment"] = True
loader_checks["Every sample appears once per epoch"] = True

for epoch, order in enumerate(epoch_orders, start=1):
    print(f"Epoch {epoch}: {order}")
print("PASS: shuffling preserves pairs and visits every sample exactly once per epoch")

Epoch 1: [1, 10, 3, 9, 8, 4, 7, 0, 6, 2, 5]
Epoch 2: [2, 5, 4, 7, 10, 6, 9, 1, 0, 3, 8]
PASS: shuffling preserves pairs and visits every sample exactly once per epoch


### Empty dataset

An empty dataset has no valid batch starts: its loader length is zero and iteration yields nothing.

In [11]:
empty_loader = DataLoader(empty_dataset, batch_size=4, shuffle=False)
empty_batches = list(empty_loader)

assert len(empty_loader) == 0
assert empty_batches == []
loader_checks["Empty dataset produces zero batches"] = True

print(f"len(empty_loader): {len(empty_loader)}")
print(f"Batches produced:   {len(empty_batches)}")
print("PASS: an empty dataset produces zero batches")

len(empty_loader): 0
Batches produced:   0
PASS: an empty dataset produces zero batches


### `batch_size` validation

A batch size must be a positive built-in integer. Zero and negative integers describe no usable group size, while floats, strings, `None`, and Booleans are the wrong type. Booleans need an explicit type check because Python otherwise treats `bool` as a subclass of `int`.

In [12]:
nonpositive_batch_sizes = [0, -1, -4]
for invalid_size in nonpositive_batch_sizes:
    assert_raises(
        ValueError,
        lambda invalid_size=invalid_size: DataLoader(batch_dataset, invalid_size),
    )

wrong_type_batch_sizes = [4.0, "4", None, True, False]
for invalid_size in wrong_type_batch_sizes:
    assert_raises(
        TypeError,
        lambda invalid_size=invalid_size: DataLoader(batch_dataset, invalid_size),
    )

loader_checks["batch_size=0 and negative values raise ValueError"] = True
loader_checks["Float, string, None, True, and False batch sizes raise TypeError"] = True

print(f"ValueError inputs: {[repr(value) for value in nonpositive_batch_sizes]}")
print(f"TypeError inputs:  {[repr(value) for value in wrong_type_batch_sizes]}")
print("PASS: batch_size accepts only positive built-in integers")

ValueError inputs: ['0', '-1', '-4']
TypeError inputs:  ['4.0', "'4'", 'None', 'True', 'False']
PASS: batch_size accepts only positive built-in integers


### `shuffle` validation

`shuffle` is a switch rather than a truthiness test, so only the built-in Boolean values `True` and `False` are accepted.

In [13]:
non_boolean_shuffle_values = [0, 1, 0.0, "yes", None, [], np.bool_(True)]
for invalid_shuffle in non_boolean_shuffle_values:
    assert_raises(
        TypeError,
        lambda invalid_shuffle=invalid_shuffle: DataLoader(
            batch_dataset, batch_size=4, shuffle=invalid_shuffle
        ),
    )

loader_checks["Non-Boolean shuffle values raise TypeError"] = True

print(f"Rejected values: {[repr(value) for value in non_boolean_shuffle_values]}")
print("PASS: shuffle accepts only True or False")

Rejected values: ['0', '1', '0.0', "'yes'", 'None', '[]', 'np.True_']
PASS: shuffle accepts only True or False


## Results

This cell checks off the requested behaviors from the values recorded by the executed assertion cells above.

In [14]:
expected_checks = [
    "11 samples with batch size 4 produces batch sizes 4, 4, 3",
    "len(loader) returns 3",
    "Batch feature and label shapes are correct",
    "shuffle=False preserves sequential order",
    "shuffle=True preserves feature-label alignment",
    "Every sample appears once per epoch",
    "Empty dataset produces zero batches",
    "batch_size=0 and negative values raise ValueError",
    "Float, string, None, True, and False batch sizes raise TypeError",
    "Non-Boolean shuffle values raise TypeError",
]

assert list(loader_checks) == expected_checks
assert all(loader_checks.values())

for check in expected_checks:
    print(f"[x] {check}")
print(f"\nAll {len(expected_checks)} requested DataLoader checks passed.")

[x] 11 samples with batch size 4 produces batch sizes 4, 4, 3
[x] len(loader) returns 3
[x] Batch feature and label shapes are correct
[x] shuffle=False preserves sequential order
[x] shuffle=True preserves feature-label alignment
[x] Every sample appears once per epoch
[x] Empty dataset produces zero batches
[x] batch_size=0 and negative values raise ValueError
[x] Float, string, None, True, and False batch sizes raise TypeError
[x] Non-Boolean shuffle values raise TypeError

All 10 requested DataLoader checks passed.
